In [30]:
# STEP 1 — Setup

import os
import time
from datetime import datetime
from dotenv import load_dotenv

from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_pinecone import PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings

load_dotenv()

# Pinecone index
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
INDEX_NAME = "coffee-hybrid"

if INDEX_NAME in pc.list_indexes().names():
    print(f"Deleting existing index: {INDEX_NAME}")
    pc.delete_index(INDEX_NAME)
    for _ in range(30):
        if INDEX_NAME not in pc.list_indexes().names():
            print("✅ Deleted.")
            break
        time.sleep(1)

# Create hybrid-ready index
pc.create_index(
    name=INDEX_NAME,
    dimension=384,
    metric="dotproduct",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

# Wait until index is ready for writes
print("⏳ Waiting for index to be ready...")
for _ in range(60):
    if pc.describe_index(INDEX_NAME).status["ready"]:
        print("✅ Index ready.")
        break
    time.sleep(1)

index = pc.Index(INDEX_NAME)

# Embeddings & LLM
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-MiniLM-L6-v2"
)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0.2,
)

# Memory namespace
USER_ID = "user_001"
MEM_NS = f"mem_{USER_ID}"

Deleting existing index: coffee-hybrid
✅ Deleted.
⏳ Waiting for index to be ready...
✅ Index ready.


In [31]:
MEM_NS

'mem_user_001'

In [32]:
# STEP 2 — Retrievers

from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_core.documents import Document

# Hybrid retriever (dense + BM25 sparse)
bm25 = BM25Encoder().default()
ret_docs_base = PineconeHybridSearchRetriever(
    embeddings=embedding,
    sparse_encoder=bm25,
    index=index,
    top_k=20,
    alpha=0.5,  # 0=sparse only, 1=dense only
    text_key="text",  # match the text_key used when storing documents
)

# Multi-query expansion
ret_docs_mq = MultiQueryRetriever.from_llm(
    retriever=ret_docs_base,
    llm=llm,
    include_original=True
)

# Memory retriever (separate namespace)
vs_mem = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embedding,
    text_key="text",
    namespace=MEM_NS
)
ret_mem = vs_mem.as_retriever(search_kwargs={"k": 6,  "namespace": MEM_NS})

# Add to memory
def add_memory(text: str, **meta):
    meta.setdefault("kind", "memory")
    meta.setdefault("ts", int(datetime.utcnow().timestamp()))
    vs_mem.add_documents([Document(page_content=text, metadata=meta)])
    print(f"  [Memory saved] \"{text[:80]}\"")

In [33]:
# STEP 2.5 — Populate Index (Load & Store Documents)

from pathlib import Path
from langchain_community.document_loaders import UnstructuredHTMLLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import re

# Function to remove boilerplate disclaimer from document content
def remove_disclaimer(text):
    """Remove common disclaimer text that appears on all pages."""
    disclaimer_pattern = r"Disclaimer:[\s\S]*?before trying new herbs or routines\.\s*"
    text = re.sub(disclaimer_pattern, "", text, flags=re.IGNORECASE)
    return text.strip()

# Load HTML documents from coffee_pages directory
html_dir = Path("coffee_pages")
html_docs = []

for fp in html_dir.glob("*.html"):
    try:
        loaded = UnstructuredHTMLLoader(str(fp)).load()
        for d in loaded:
            meta = dict(d.metadata or {})
            meta.pop("text", None)
            meta["source"] = str(fp)
            
            # Remove disclaimer from content
            content = remove_disclaimer(d.page_content)
            
            if content:  # Only keep if there's meaningful content left
                html_docs.append(Document(page_content=content, metadata=meta))
    except Exception as e:
        print(f"[WARN] Skipping {fp.name}: {e}")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
chunks = splitter.split_documents(html_docs)

# Store in Pinecone using PineconeVectorStore.from_documents
vectorstore = PineconeVectorStore.from_documents(
    chunks,
    embedding=embedding,
    index_name=INDEX_NAME,
    text_key="text",
)

print(f"✅ Stored {len(chunks)} chunks in '{INDEX_NAME}' (disclaimer removed)")

✅ Stored 53 chunks in 'coffee-hybrid' (disclaimer removed)


In [34]:
# STEP 3 — Fusion + Deduplication

from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain.retrievers.document_compressors import DocumentCompressorPipeline

# Combine docs and memory with weighted fusion
ret_fused = EnsembleRetriever(
    retrievers=[ret_docs_mq, ret_mem],
    weights=[0.25, 0.75],  # prioritize docs, include memory
)

# Deduplicate similar chunks
dedup = EmbeddingsRedundantFilter(
    embeddings=embedding,
    similarity_threshold=0.92
)

ret_final = ContextualCompressionRetriever(
    base_retriever=ret_fused,
    base_compressor=DocumentCompressorPipeline(transformers=[dedup]),

)

In [35]:
# STEP 4 — Prompt + Helpers

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import HumanMessage, AIMessage

def format_context(docs):
    """Format retrieved docs as [S#] source | snippet."""
    lines = []
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source") or d.metadata.get("kind") or f"[S{i}]"
        text = (d.page_content or "").replace("\n", " ")
        lines.append(f"[S{i}] {src}\n{text}")
    return "\n\n".join(lines)

def cap_docs(docs, max_chars=6000):
    """Keep docs up to max character limit."""
    kept, total = [], 0
    for d in docs:
        chars = len(d.page_content or "")
        if total + chars > max_chars:
            break
        kept.append(d)
        total += chars
    return kept

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are Askly, a helpful assistant. "
     "Answer ONLY using CONTEXT. If missing info, say so. Cite [S#]."
    ),
    MessagesPlaceholder("history"),
    ("system", "CONTEXT:\n{context}"),
    ("human", "{question}")
])


answer_chain = prompt | llm | StrOutputParser()

In [ ]:
# STEP 5 — Memory-RAG Demo (scripted conversation)
#
# Each entry is either:
#   ("ask", "<question>")   — retrieve + answer, history grows
#   ("mem", "<fact>")       — persist a fact to the memory namespace
#
# The instrumentation prints show exactly how history and memory
# accumulate turn-by-turn.

from langchain.schema import HumanMessage, AIMessage

DEMO = [
    # Turn 1 — cold start, no history, no memory
    ("ask", "What are the contents of ashwagandha coffee?"),

    # Turn 2 — follow-up; LLM must use history to know what "those benefits" refers to
    ("ask", "How should I drink ashwagandha coffee?"),

    # Persist a user preference to the memory namespace
    ("mem", "I prefer caffeine-light options because I'm sensitive to caffeine"),

    # Turn 3 — new topic; memory retriever should surface the saved preference
    ("ask", "Can you suggest a good morning coffee ritual for me?"),

    # Turn 4 — another follow-up; tests both history retention AND memory retrieval
    ("ask", "Would mushroom coffee be a good fit given what you know about me?"),

    # Turn 5 — synthesis; model must draw on 4 prior turns + saved memory
    ("ask", "Summarise everything we discussed and give me one concrete recommendation."),
]

# ── run ──────────────────────────────────────────────────────────────────────

history = []
print("🤖  Askly — Memory-RAG scripted demo")
print("=" * 60)

for kind, text in DEMO:

    # ── /mem command ─────────────────────────────────────────────
    if kind == "mem":
        print(f"\n📌  Saving to persistent memory: \"{text}\"")
        add_memory(text)
        print()
        continue

    # ── ask ──────────────────────────────────────────────────────
    turn = len(history) // 2 + 1
    print(f"\n{'═' * 60}")
    print(f"  Turn {turn}  |  [History] {len(history)} message(s) in context")
    print(f"  Q: {text}")
    print(f"{'─' * 60}")

    history.append(HumanMessage(content=text))

    # retrieve (corpus + memory fused & deduped)
    docs = ret_final.invoke(text)
    # print(docs.page_content)
    mem_docs    = [d for d in docs if d.metadata.get("kind") == "memory"]
    corpus_docs = [d for d in docs if d.metadata.get("kind") != "memory"]
    print(f"  [Retrieval]  {len(docs)} docs → {len(corpus_docs)} corpus, {len(mem_docs)} memory")

    docs = cap_docs(docs, max_chars=6000)
    print(f"  [Capped]     {len(docs)} docs kept  ({sum(len(d.page_content) for d in docs)} chars to LLM)")
    
    answer = answer_chain.invoke({
        "history": history,
        "context": format_context(docs),
        "question": text,
    })
    history.append(AIMessage(content=answer))

    print(f"\n  A: {answer}")

    # history snapshot
    print(f"\n  [History snapshot — {len(history)} messages]")
    for msg in history:
        role    = "You  " if isinstance(msg, HumanMessage) else "Askly"
        snippet = msg.content[:80].replace("\n", " ")
        suffix  = "..." if len(msg.content) > 80 else ""
        print(f"    {role}: {snippet}{suffix}")

print(f"\n{'=' * 60}")
print("Demo complete.")

🤖  Askly — Memory-RAG scripted demo

════════════════════════════════════════════════════════════
  Turn 1  |  [History] 0 message(s) in context
  Q: What are the contents of ashwagandha coffee?
────────────────────────────────────────────────────────────
  [DEBUG ret_mem] returned 0 docs
  [Retrieval]  31 docs → 31 corpus, 0 memory
  [Capped]     13 docs kept  (5978 chars to LLM)

  A: Ashwagandha coffee contains roasted coffee combined with powdered ashwagandha. It can also include milk of choice (dairy or plant-based), and optional ingredients such as cinnamon or cardamom, sweetener (jaggery, honey, or sugar), and a pinch of black pepper [S8, S3]. Some variations may also include mushroom powders like lion's mane or chaga [S4].

  [History snapshot — 2 messages]
    You  : What are the contents of ashwagandha coffee?
    Askly: Ashwagandha coffee contains roasted coffee combined with powdered ashwagandha. I...

════════════════════════════════════════════════════════════
  Turn 2  |